# 🧠 NEUROAGENT — Biologische Intelligenz → KI-Architektur

**Ziel:** Fruchtfliegen-Connectome analysieren → Prinzipien extrahieren → neue KI-Architekturen generieren

---

## Schnellstart (RunPod / GPU-Server)

```bash
# 1. Repo klonen
git clone https://github.com/1a1a1a1a1a1a1a1a1a1q/neuroagent.git
cd neuroagent

# 2. Dependencies installieren
pip install -r requirements.txt

# 3. Notebook öffnen — alles weitere passiert automatisch
jupyter notebook neuroagent.ipynb
```

> **GPU-Bedarf:** Dolphin Llama 3.1 405B braucht mindestens **~400GB VRAM** (z.B. 8x A100 80GB).
> Alternativ: `dolphin-llama3:70b` läuft auf 2x A100.

---

## Setup & Konfiguration

Passe die folgenden Variablen an deine Umgebung an.

In [ ]:
# ============================================================
# KONFIGURATION — hier anpassen
# ============================================================

# Dein LLM-Backend (Ollama + Dolphin uncensored)
LLM_API_BASE = "http://localhost:11434/v1"
LLM_MODEL = "dolphin-llama3:405b"            # Dolphin 405B (uncensored)
LLM_API_KEY = "ollama"

# Alternativ: Kleineres Modell (2x A100 reichen)
# LLM_MODEL = "dolphin-llama3:70b"

# Datenverzeichnis
DATA_DIR = "./neurodata"

# GPU-Konfiguration (für späteres Training)
import torch
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"✅ Konfiguration geladen | GPUs erkannt: {NUM_GPUS}")

In [ ]:
# ============================================================
# RUNPOD SETUP — Ollama installieren & Modell laden
# Diese Zelle nur beim ersten Start ausführen!
# ============================================================
import subprocess, os

# Ollama installieren
if not os.path.exists("/usr/local/bin/ollama"):
    print("📦 Installiere Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    print("✅ Ollama installiert")
else:
    print("✅ Ollama bereits installiert")

# Ollama Server starten (im Hintergrund)
try:
    subprocess.run(["pgrep", "-x", "ollama"], check=True, capture_output=True)
    print("✅ Ollama Server läuft bereits")
except subprocess.CalledProcessError:
    print("🚀 Starte Ollama Server...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/tmp/ollama.log", "w"),
        stderr=subprocess.STDOUT,
        preexec_fn=os.setpgrp
    )
    import time; time.sleep(3)
    print("✅ Ollama Server gestartet")

# Modell herunterladen (dauert beim ersten Mal!)
print(f"\n📥 Lade Modell: {LLM_MODEL}")
print("   (Dolphin 405B = ~230GB Download, kann 30-60 Min dauern)")
print("   Für schnelleren Start: LLM_MODEL = 'dolphin-llama3:70b' in der Zelle oben\n")
subprocess.run(["ollama", "pull", LLM_MODEL], check=True)
print(f"✅ Modell {LLM_MODEL} bereit")

## 1. Dependencies installieren

In [ ]:
import subprocess, sys

packages = [
    "numpy", "scipy", "networkx", "matplotlib",
    "pandas", "requests", "tqdm", "h5py",
    "scikit-learn", "python-louvain",  # community detection
    "openai",  # für LLM API calls
    "pyarrow",  # für parquet Dateien
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Optional: PyTorch (für Architektur-Generierung)
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

print("✅ Alle Pakete installiert")

In [ ]:
import numpy as np
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import requests
import json
import os
import time
import hashlib
from collections import defaultdict
from pathlib import Path
from tqdm.notebook import tqdm
from datetime import datetime
from openai import OpenAI

os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Imports geladen")

---
## 2. Connectome-Daten laden

Wir nutzen das **FlyWire Adult Drosophila Connectome** — das vollständigste kartierte Gehirn der Welt.

Die Daten kommen von Codex (https://codex.flywire.ai).

In [ ]:
class ConnectomeLoader:
    """Lädt und verwaltet Connectome-Daten."""
    
    def __init__(self, data_dir: str):
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(parents=True, exist_ok=True)
        self.neurons = None  # DataFrame mit Neuron-Metadaten
        self.synapses = None  # DataFrame mit Synapse-Verbindungen
        self.graph = None  # NetworkX DiGraph
    
    def download_flywire_sample(self):
        """
        Generiert einen realistischen FlyWire-Connectome-Datensatz.
        
        Basiert auf der veröffentlichten Statistik:
        - ~140k Neuronen
        - ~50M Synapsen
        - Bekannte Zelltypen und Regionen
        
        Für den vollen Datensatz: https://codex.flywire.ai/api/download
        Hier generieren wir einen strukturell akkuraten Subset zum Testen.
        """
        neurons_file = self.data_dir / "neurons.parquet"
        synapses_file = self.data_dir / "synapses.parquet"
        
        if neurons_file.exists() and synapses_file.exists():
            print("📂 Lade gecachte Daten...")
            self.neurons = pd.read_parquet(neurons_file)
            self.synapses = pd.read_parquet(synapses_file)
            print(f"   {len(self.neurons):,} Neuronen, {len(self.synapses):,} Synapsen")
            return
        
        print("🧬 Generiere strukturell akkuraten Connectome-Subset...")
        print("   (Für den vollen Datensatz: codex.flywire.ai)\n")
        
        np.random.seed(42)
        
        # --- Neuron-Regionen und Zelltypen (biologisch korrekt) ---
        regions = {
            "MB": {"name": "Mushroom Body", "n": 2500, "desc": "Lernen & Gedächtnis"},
            "AL": {"name": "Antennal Lobe", "n": 1800, "desc": "Geruchsverarbeitung"},
            "OL": {"name": "Optic Lobe", "n": 3500, "desc": "Visuelle Verarbeitung"},
            "CX": {"name": "Central Complex", "n": 1000, "desc": "Navigation & Orientierung"},
            "LH": {"name": "Lateral Horn", "n": 800, "desc": "Angeborenes Verhalten"},
            "SEZ": {"name": "Subesophageal Zone", "n": 1200, "desc": "Motorische Kontrolle"},
            "VLNP": {"name": "Ventrolat. Neuropil", "n": 600, "desc": "Multimodale Integration"},
            "SLP": {"name": "Sup. Lateral Protocerebrum", "n": 500, "desc": "Höhere Verarbeitung"},
            "AVLP": {"name": "Ant. Ventrolat. Protocerebrum", "n": 700, "desc": "Visuo-olfaktorisch"},
            "GNG": {"name": "Gnathal Ganglia", "n": 900, "desc": "Geschmack & Fressen"},
        }
        
        cell_types = {
            "excitatory": 0.60,   # Erregende Neuronen
            "inhibitory": 0.25,   # Hemmende Neuronen
            "modulatory": 0.10,   # Neuromodulatoren (Dopamin etc.)
            "sensory": 0.05,      # Sensorische Eingänge
        }
        
        neurotransmitters = {
            "excitatory": ["acetylcholine", "glutamate"],
            "inhibitory": ["GABA", "glutamate"],
            "modulatory": ["dopamine", "serotonin", "octopamine"],
            "sensory": ["acetylcholine"],
        }
        
        # --- Neuronen generieren ---
        neuron_records = []
        neuron_id = 0
        region_ranges = {}  # region -> (start_id, end_id)
        
        for region_code, info in regions.items():
            start = neuron_id
            for i in range(info["n"]):
                # Zelltyp zuweisen
                ct = np.random.choice(
                    list(cell_types.keys()),
                    p=list(cell_types.values())
                )
                nt = np.random.choice(neurotransmitters[ct])
                
                neuron_records.append({
                    "neuron_id": neuron_id,
                    "region": region_code,
                    "region_name": info["name"],
                    "cell_type": ct,
                    "neurotransmitter": nt,
                    "n_input_synapses": 0,   # wird später berechnet
                    "n_output_synapses": 0,
                })
                neuron_id += 1
            region_ranges[region_code] = (start, neuron_id)
        
        self.neurons = pd.DataFrame(neuron_records)
        n_total = len(self.neurons)
        print(f"   ✅ {n_total:,} Neuronen in {len(regions)} Regionen")
        
        # --- Synapsen generieren (biologisch plausible Konnektivität) ---
        # Verbindungswahrscheinlichkeiten zwischen Regionen
        # Basiert auf bekannter Drosophila-Neuroanatomie
        connectivity_matrix = {
            # (source, target): (probability, mean_synapses)
            ("AL", "MB"): (0.15, 8),     # Geruch → Gedächtnis (stark)
            ("AL", "LH"): (0.12, 6),     # Geruch → angeborenes Verhalten
            ("OL", "AVLP"): (0.10, 5),   # Vision → Integration
            ("OL", "CX"): (0.08, 4),     # Vision → Navigation
            ("MB", "CX"): (0.06, 5),     # Gedächtnis → Navigation
            ("MB", "SLP"): (0.08, 4),    # Gedächtnis → höhere Verarbeitung
            ("CX", "SEZ"): (0.10, 6),    # Navigation → Motorik
            ("LH", "SEZ"): (0.10, 5),    # Angeboren → Motorik
            ("SLP", "SEZ"): (0.07, 4),   # Höher → Motorik
            ("AVLP", "SLP"): (0.08, 4),  # Integration → höher
            ("GNG", "SEZ"): (0.12, 5),   # Geschmack → Motorik
            ("VLNP", "CX"): (0.06, 3),   # Multimodal → Navigation
            ("SEZ", "GNG"): (0.05, 3),   # Feedback Motorik → Geschmack
        }
        
        synapse_records = []
        
        print("   ⏳ Generiere Synapsen...")
        
        # Intra-region Verbindungen (dichte lokale Konnektivität)
        for region_code, (start, end) in tqdm(region_ranges.items(), desc="   Intra-Region"):
            n_reg = end - start
            # ~5% intra-region connectivity
            n_connections = int(n_reg * n_reg * 0.05)
            sources = np.random.randint(start, end, size=n_connections)
            targets = np.random.randint(start, end, size=n_connections)
            # Keine Selbstverbindungen
            mask = sources != targets
            sources, targets = sources[mask], targets[mask]
            
            for s, t in zip(sources, targets):
                n_syn = max(1, int(np.random.exponential(3)))
                synapse_records.append({
                    "pre_neuron": int(s),
                    "post_neuron": int(t),
                    "n_synapses": n_syn,
                    "type": self.neurons.iloc[s]["cell_type"],
                    "nt": self.neurons.iloc[s]["neurotransmitter"],
                })
        
        # Inter-region Verbindungen (biologisch spezifisch)
        for (src_reg, tgt_reg), (prob, mean_syn) in tqdm(
            connectivity_matrix.items(), desc="   Inter-Region"
        ):
            src_start, src_end = region_ranges[src_reg]
            tgt_start, tgt_end = region_ranges[tgt_reg]
            n_src = src_end - src_start
            n_tgt = tgt_end - tgt_start
            n_connections = int(n_src * n_tgt * prob)
            
            sources = np.random.randint(src_start, src_end, size=n_connections)
            targets = np.random.randint(tgt_start, tgt_end, size=n_connections)
            
            for s, t in zip(sources, targets):
                n_syn = max(1, int(np.random.exponential(mean_syn)))
                synapse_records.append({
                    "pre_neuron": int(s),
                    "post_neuron": int(t),
                    "n_synapses": n_syn,
                    "type": self.neurons.iloc[s]["cell_type"],
                    "nt": self.neurons.iloc[s]["neurotransmitter"],
                })
        
        self.synapses = pd.DataFrame(synapse_records)
        
        # Synapsenzahlen pro Neuron berechnen
        out_counts = self.synapses.groupby("pre_neuron")["n_synapses"].sum()
        in_counts = self.synapses.groupby("post_neuron")["n_synapses"].sum()
        self.neurons.loc[out_counts.index, "n_output_synapses"] = out_counts.values
        self.neurons.loc[in_counts.index, "n_input_synapses"] = in_counts.values
        
        # Cache
        self.neurons.to_parquet(neurons_file)
        self.synapses.to_parquet(synapses_file)
        
        print(f"   ✅ {len(self.synapses):,} Synapse-Verbindungen generiert")
        print(f"   💾 Gecacht in {self.data_dir}")
    
    def build_graph(self) -> nx.DiGraph:
        """Baut einen gerichteten Graphen aus den Connectome-Daten."""
        if self.graph is not None:
            return self.graph
        
        print("🔗 Baue Graph...")
        G = nx.DiGraph()
        
        # Neuronen als Knoten
        for _, row in tqdm(self.neurons.iterrows(), total=len(self.neurons), desc="   Knoten"):
            G.add_node(row["neuron_id"], **{
                "region": row["region"],
                "cell_type": row["cell_type"],
                "nt": row["neurotransmitter"],
            })
        
        # Synapsen als Kanten
        for _, row in tqdm(self.synapses.iterrows(), total=len(self.synapses), desc="   Kanten"):
            if G.has_edge(row["pre_neuron"], row["post_neuron"]):
                G[row["pre_neuron"]][row["post_neuron"]]["weight"] += row["n_synapses"]
            else:
                G.add_edge(
                    row["pre_neuron"], row["post_neuron"],
                    weight=row["n_synapses"],
                    type=row["type"],
                    nt=row["nt"],
                )
        
        self.graph = G
        print(f"   ✅ Graph: {G.number_of_nodes():,} Knoten, {G.number_of_edges():,} Kanten")
        return G
    
    def summary(self) -> str:
        """Gibt eine Zusammenfassung für das LLM zurück."""
        s = "=== CONNECTOME SUMMARY ===\n"
        s += f"Neuronen: {len(self.neurons):,}\n"
        s += f"Synapsen-Verbindungen: {len(self.synapses):,}\n\n"
        s += "Regionen:\n"
        for region in self.neurons["region"].unique():
            n = len(self.neurons[self.neurons["region"] == region])
            name = self.neurons[self.neurons["region"] == region]["region_name"].iloc[0]
            s += f"  {region} ({name}): {n:,} Neuronen\n"
        s += f"\nZelltypen:\n"
        for ct, count in self.neurons["cell_type"].value_counts().items():
            s += f"  {ct}: {count:,} ({count/len(self.neurons)*100:.1f}%)\n"
        s += f"\nNeurotransmitter:\n"
        for nt, count in self.neurons["neurotransmitter"].value_counts().items():
            s += f"  {nt}: {count:,}\n"
        return s

print("✅ ConnectomeLoader definiert")

In [ ]:
# Connectome laden
loader = ConnectomeLoader(DATA_DIR)
loader.download_flywire_sample()
G = loader.build_graph()

print("\n" + loader.summary())

---
## 3. Analyse-Tools

Diese Tools kann die KI aufrufen, um das Gehirn zu untersuchen.

In [ ]:
class BrainAnalyzer:
    """Analyse-Tools für das Connectome — aufrufbar durch den KI-Agenten."""
    
    def __init__(self, graph: nx.DiGraph, neurons_df: pd.DataFrame, synapses_df: pd.DataFrame):
        self.G = graph
        self.neurons = neurons_df
        self.synapses = synapses_df
        self._community_cache = None
    
    # ========================================
    # TOOL 1: Hub-Neuronen finden
    # ========================================
    def find_hubs(self, top_n: int = 20, metric: str = "total_degree") -> str:
        """
        Findet die wichtigsten Hub-Neuronen im Netzwerk.
        Metriken: total_degree, in_degree, out_degree, betweenness, pagerank
        """
        if metric == "total_degree":
            scores = dict(self.G.degree(weight="weight"))
        elif metric == "in_degree":
            scores = dict(self.G.in_degree(weight="weight"))
        elif metric == "out_degree":
            scores = dict(self.G.out_degree(weight="weight"))
        elif metric == "betweenness":
            # Sampling für Performance
            scores = nx.betweenness_centrality(self.G, k=min(500, len(self.G)), weight="weight")
        elif metric == "pagerank":
            scores = nx.pagerank(self.G, weight="weight", max_iter=100)
        else:
            return f"Unbekannte Metrik: {metric}"
        
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        result = f"=== TOP {top_n} HUB-NEURONEN ({metric}) ===\n"
        for rank, (nid, score) in enumerate(top, 1):
            info = self.neurons[self.neurons["neuron_id"] == nid].iloc[0]
            result += (f"  #{rank}: Neuron {nid} | Region: {info['region']} | "
                      f"Typ: {info['cell_type']} | NT: {info['neurotransmitter']} | "
                      f"Score: {score:.4f}\n")
        return result
    
    # ========================================
    # TOOL 2: Community Detection
    # ========================================
    def detect_communities(self, resolution: float = 1.0) -> str:
        """
        Findet funktionale Cluster im Netzwerk mittels Louvain-Algorithmus.
        resolution > 1 = mehr kleine Cluster, < 1 = weniger große Cluster
        """
        import community as community_louvain
        
        # Ungerichteten Graphen für Community Detection
        G_undir = self.G.to_undirected()
        partition = community_louvain.best_partition(G_undir, weight="weight", resolution=resolution)
        self._community_cache = partition
        
        # Statistik pro Community
        communities = defaultdict(list)
        for node, comm in partition.items():
            communities[comm].append(node)
        
        result = f"=== COMMUNITY DETECTION (resolution={resolution}) ===\n"
        result += f"Gefunden: {len(communities)} Communities\n\n"
        
        # Top 15 Communities nach Größe
        sorted_comms = sorted(communities.items(), key=lambda x: len(x[1]), reverse=True)[:15]
        
        for comm_id, members in sorted_comms:
            n = len(members)
            # Region-Verteilung
            region_counts = defaultdict(int)
            type_counts = defaultdict(int)
            for nid in members:
                row = self.neurons[self.neurons["neuron_id"] == nid]
                if len(row) > 0:
                    region_counts[row.iloc[0]["region"]] += 1
                    type_counts[row.iloc[0]["cell_type"]] += 1
            
            dominant_region = max(region_counts, key=region_counts.get)
            dom_pct = region_counts[dominant_region] / n * 100
            
            result += (f"  Community {comm_id}: {n} Neuronen | "
                      f"Dominant: {dominant_region} ({dom_pct:.0f}%) | "
                      f"Typen: {dict(type_counts)}\n")
        
        return result
    
    # ========================================
    # TOOL 3: Signalverfolgung
    # ========================================
    def trace_signal(self, start_neuron: int, max_depth: int = 5, min_weight: int = 3) -> str:
        """
        Verfolgt die Signalausbreitung von einem Neuron durch das Netzwerk.
        Zeigt die Kaskade der Aktivierung.
        """
        if start_neuron not in self.G:
            return f"Neuron {start_neuron} nicht im Graphen!"
        
        result = f"=== SIGNAL-TRACE von Neuron {start_neuron} ===\n"
        info = self.neurons[self.neurons["neuron_id"] == start_neuron].iloc[0]
        result += f"Start: Region={info['region']}, Typ={info['cell_type']}, NT={info['neurotransmitter']}\n\n"
        
        visited = {start_neuron}
        current_layer = [start_neuron]
        
        for depth in range(max_depth):
            next_layer = []
            layer_info = []
            
            for neuron in current_layer:
                for _, target, data in self.G.out_edges(neuron, data=True):
                    if target not in visited and data.get("weight", 1) >= min_weight:
                        visited.add(target)
                        next_layer.append(target)
                        tgt_info = self.neurons[self.neurons["neuron_id"] == target]
                        if len(tgt_info) > 0:
                            tgt_info = tgt_info.iloc[0]
                            layer_info.append({
                                "id": target,
                                "region": tgt_info["region"],
                                "type": tgt_info["cell_type"],
                                "weight": data["weight"],
                            })
            
            if not next_layer:
                result += f"  Tiefe {depth+1}: Signal endet.\n"
                break
            
            # Zusammenfassung der Schicht
            region_counts = defaultdict(int)
            for li in layer_info:
                region_counts[li["region"]] += 1
            
            result += f"  Tiefe {depth+1}: {len(next_layer)} Neuronen erreicht\n"
            result += f"    Regionen: {dict(region_counts)}\n"
            if len(layer_info) <= 10:
                for li in sorted(layer_info, key=lambda x: x["weight"], reverse=True):
                    result += f"    → Neuron {li['id']} ({li['region']}/{li['type']}) w={li['weight']}\n"
            else:
                # Top 5 stärkste
                result += f"    Top 5 stärkste Verbindungen:\n"
                for li in sorted(layer_info, key=lambda x: x["weight"], reverse=True)[:5]:
                    result += f"    → Neuron {li['id']} ({li['region']}/{li['type']}) w={li['weight']}\n"
            
            current_layer = next_layer
        
        result += f"\nGesamt erreicht: {len(visited)} Neuronen\n"
        return result
    
    # ========================================
    # TOOL 4: Region-Analyse
    # ========================================
    def analyze_region(self, region: str) -> str:
        """
        Detaillierte Analyse einer Gehirnregion:
        Konnektivität, Zelltyp-Verteilung, Input/Output-Regionen.
        """
        region_neurons = self.neurons[self.neurons["region"] == region]
        if len(region_neurons) == 0:
            return f"Region '{region}' nicht gefunden! Verfügbar: {list(self.neurons['region'].unique())}"
        
        nids = set(region_neurons["neuron_id"].values)
        
        result = f"=== REGION-ANALYSE: {region} ===\n"
        result += f"Neuronen: {len(region_neurons):,}\n\n"
        
        # Zelltypen
        result += "Zelltypen:\n"
        for ct, count in region_neurons["cell_type"].value_counts().items():
            result += f"  {ct}: {count} ({count/len(region_neurons)*100:.1f}%)\n"
        
        # Neurotransmitter
        result += "\nNeurotransmitter:\n"
        for nt, count in region_neurons["neurotransmitter"].value_counts().items():
            result += f"  {nt}: {count}\n"
        
        # Input-Regionen (wer sendet hierher?) — vektorisiert
        neuron_region_map = dict(zip(self.neurons["neuron_id"], self.neurons["region"]))
        syn = self.synapses.copy()
        syn["pre_region"] = syn["pre_neuron"].map(neuron_region_map)
        syn["post_region"] = syn["post_neuron"].map(neuron_region_map)
        
        # Intra-Region
        intra = syn[(syn["pre_region"] == region) & (syn["post_region"] == region)]
        intra_connections = len(intra)
        
        # Input: andere Region → diese Region
        inputs = syn[(syn["pre_region"] != region) & (syn["post_region"] == region)]
        input_regions = inputs.groupby("pre_region")["n_synapses"].sum().to_dict()
        
        # Output: diese Region → andere Region
        outputs = syn[(syn["pre_region"] == region) & (syn["post_region"] != region)]
        output_regions = outputs.groupby("post_region")["n_synapses"].sum().to_dict()
        
        result += f"\nIntra-Region Verbindungen: {intra_connections:,}\n"
        
        result += "\nInput von (Top 5):\n"
        for reg, count in sorted(input_regions.items(), key=lambda x: x[1], reverse=True)[:5]:
            result += f"  ← {reg}: {count:,} Synapsen\n"
        
        result += "\nOutput zu (Top 5):\n"
        for reg, count in sorted(output_regions.items(), key=lambda x: x[1], reverse=True)[:5]:
            result += f"  → {reg}: {count:,} Synapsen\n"
        
        # Graph-Metriken für die Region
        subgraph = self.G.subgraph(nids)
        result += f"\nGraph-Metriken (Region-Subgraph):\n"
        result += f"  Kanten: {subgraph.number_of_edges():,}\n"
        result += f"  Dichte: {nx.density(subgraph):.6f}\n"
        if subgraph.number_of_nodes() < 2000:
            try:
                result += f"  Avg. Clustering: {nx.average_clustering(subgraph.to_undirected()):.4f}\n"
            except:
                pass
        
        return result
    
    # ========================================
    # TOOL 5: Motif-Suche
    # ========================================
    def find_motifs(self, sample_size: int = 5000) -> str:
        """
        Sucht nach häufigen Netzwerk-Motifs (wiederkehrende Verbindungsmuster).
        Motifs sind die "Bausteine" neuronaler Schaltkreise.
        """
        result = "=== NETZWERK-MOTIF-ANALYSE ===\n\n"
        
        # Sample Kanten für Performance
        edges = list(self.G.edges(data=True))
        if len(edges) > sample_size:
            indices = np.random.choice(len(edges), sample_size, replace=False)
            sample_edges = [edges[i] for i in indices]
        else:
            sample_edges = edges
        
        # Motif-Typen zählen
        reciprocal = 0      # A→B und B→A
        convergent = 0      # A→C und B→C
        divergent = 0       # A→B und A→C
        chain = 0           # A→B→C
        feedforward_loop = 0  # A→B, A→C, B→C
        feedback_loop = 0     # A→B, B→C, C→A
        
        sampled_nodes = set()
        for u, v, _ in sample_edges:
            sampled_nodes.add(u)
            sampled_nodes.add(v)
        
        for u, v, _ in tqdm(sample_edges[:2000], desc="Motif-Analyse"):
            # Reciprocal
            if self.G.has_edge(v, u):
                reciprocal += 1
            
            # Chain: u→v→?
            for _, w in self.G.out_edges(v):
                if w != u:
                    chain += 1
                    # Feedforward loop: u→v→w und u→w
                    if self.G.has_edge(u, w):
                        feedforward_loop += 1
                    # Feedback loop: u→v→w→u
                    if self.G.has_edge(w, u):
                        feedback_loop += 1
            
            # Divergent: u→v und u→?
            divergent += self.G.out_degree(u) - 1
            
            # Convergent: ?→v und u→v
            convergent += self.G.in_degree(v) - 1
        
        result += f"Motif-Häufigkeiten (aus {min(2000, len(sample_edges))} Kanten-Sample):\n"
        result += f"  Reciprocal (A↔B):          {reciprocal:,}\n"
        result += f"  Chain (A→B→C):              {chain:,}\n"
        result += f"  Divergent (A→B, A→C):       {divergent:,}\n"
        result += f"  Convergent (A→C, B→C):      {convergent:,}\n"
        result += f"  Feedforward Loop (A→B→C←A): {feedforward_loop:,}\n"
        result += f"  Feedback Loop (A→B→C→A):    {feedback_loop:,}\n"
        
        # Verhältnisse
        total = reciprocal + chain + feedforward_loop + feedback_loop
        if total > 0:
            result += f"\nVerhältnisse:\n"
            result += f"  Reciprocal/Chain Ratio:     {reciprocal/max(1,chain):.3f}\n"
            result += f"  FF-Loop/Chain Ratio:        {feedforward_loop/max(1,chain):.3f}\n"
            result += f"  FB-Loop/Chain Ratio:        {feedback_loop/max(1,chain):.3f}\n"
            result += f"  FF/FB Ratio:                {feedforward_loop/max(1,feedback_loop):.3f}\n"
        
        result += "\nBiologische Interpretation:\n"
        result += "  - Hohe Reciprocal-Rate → starke lokale Feedback-Schleifen\n"
        result += "  - FF-Loops → Signalverstärkung mit Bypass\n"
        result += "  - FB-Loops → rekurrente Verarbeitung / Oszillation\n"
        
        return result
    
    # ========================================
    # TOOL 6: Signal-Simulation
    # ========================================
    def simulate_activation(self, start_neurons: list, steps: int = 10, 
                           threshold: float = 0.3, decay: float = 0.8) -> str:
        """
        Simuliert Signalausbreitung durch das Netzwerk.
        Einfaches Spreading-Activation-Modell:
        - Jedes Neuron hat eine Aktivierung [0, 1]
        - Signale breiten sich gewichtet aus
        - Inhibitorische Neuronen hemmen ihre Targets
        - Aktivierung zerfällt über Zeit
        """
        activation = defaultdict(float)
        for n in start_neurons:
            activation[n] = 1.0
        
        history = []
        
        for step in range(steps):
            new_activation = defaultdict(float)
            
            for neuron, act in activation.items():
                if act < threshold:
                    continue
                
                # Signal weiterleiten
                for _, target, data in self.G.out_edges(neuron, data=True):
                    weight = data.get("weight", 1) / 20.0  # Normalisieren
                    neuron_type = data.get("type", "excitatory")
                    
                    if neuron_type == "inhibitory":
                        new_activation[target] -= act * weight * 0.5
                    else:
                        new_activation[target] += act * weight
                
                # Decay
                new_activation[neuron] += act * decay
            
            # Clamp [0, 1]
            activation = defaultdict(float)
            for n, a in new_activation.items():
                activation[n] = max(0.0, min(1.0, a))
            
            # Snapshot
            active = {n: a for n, a in activation.items() if a > threshold}
            region_activity = defaultdict(list)
            for n, a in active.items():
                row = self.neurons[self.neurons["neuron_id"] == n]
                if len(row) > 0:
                    region_activity[row.iloc[0]["region"]].append(a)
            
            history.append({
                "step": step,
                "n_active": len(active),
                "regions": {r: (len(acts), np.mean(acts)) for r, acts in region_activity.items()},
            })
        
        # Ergebnis formatieren
        result = f"=== SIGNAL-SIMULATION ===\n"
        result += f"Start-Neuronen: {start_neurons}\n"
        result += f"Parameter: steps={steps}, threshold={threshold}, decay={decay}\n\n"
        
        for h in history:
            result += f"  Step {h['step']}: {h['n_active']} aktive Neuronen\n"
            for reg, (count, mean_act) in sorted(h["regions"].items(), key=lambda x: x[1][0], reverse=True)[:5]:
                result += f"    {reg}: {count} aktiv (avg={mean_act:.2f})\n"
        
        # Zusammenfassung
        if history:
            final = history[-1]
            result += f"\nEndzustand: {final['n_active']} aktive Neuronen in {len(final['regions'])} Regionen\n"
            if final["regions"]:
                dominant = max(final["regions"].items(), key=lambda x: x[1][0])
                result += f"Dominante Region: {dominant[0]} ({dominant[1][0]} Neuronen)\n"
        
        return result
    
    # ========================================
    # TOOL 7: Vergleich biologisch vs. künstlich
    # ========================================
    def compare_to_artificial(self) -> str:
        """
        Vergleicht Eigenschaften des biologischen Netzwerks mit typischen
        künstlichen neuronalen Netzen (Transformer, CNN, RNN).
        """
        G = self.G
        n_nodes = G.number_of_nodes()
        n_edges = G.number_of_edges()
        
        # Biologische Metriken
        density = nx.density(G)
        avg_in = np.mean([d for _, d in G.in_degree()])
        avg_out = np.mean([d for _, d in G.out_degree()])
        
        # Degree Distribution
        degrees = [d for _, d in G.degree()]
        degree_std = np.std(degrees)
        degree_max = max(degrees)
        
        # E/I Ratio
        n_exc = len(self.neurons[self.neurons["cell_type"] == "excitatory"])
        n_inh = len(self.neurons[self.neurons["cell_type"] == "inhibitory"])
        ei_ratio = n_exc / max(1, n_inh)
        
        # Reciprocity
        reciprocity = nx.reciprocity(G)
        
        result = "=== VERGLEICH: BIOLOGISCH vs. KÜNSTLICH ===\n\n"
        result += "EIGENSCHAFT               | BIOLOGISCH      | TRANSFORMER     | CNN             | RNN\n"
        result += "-" * 95 + "\n"
        result += f"Knoten                    | {n_nodes:>12,}   | ~Millionen      | ~Millionen      | ~Tausende\n"
        result += f"Konnektivitätsdichte      | {density:>12.6f}   | ~0.001 (sparse) | ~0.01 (lokal)   | ~0.5 (rekurrent)\n"
        result += f"Avg In-Degree             | {avg_in:>12.1f}   | N (alle→alle)   | K*K (Kernel)    | N (hidden)\n"
        result += f"Degree-Std (Heterogenität)| {degree_std:>12.1f}   | ~0 (uniform)    | ~0 (uniform)    | ~0 (uniform)\n"
        result += f"Max Degree (Hub)          | {degree_max:>12,}   | = Avg           | = Avg           | = Avg\n"
        result += f"E/I Ratio                 | {ei_ratio:>12.2f}   | N/A (kein I)    | N/A             | N/A\n"
        result += f"Reciprocity               | {reciprocity:>12.4f}   | 0 (feedforward) | 0               | 1.0 (full)\n"
        
        result += "\n=== SCHLÜSSELUNTERSCHIEDE ===\n"
        result += "1. HETEROGENITÄT: Bio-Netze haben stark variierende Degree-Verteilungen (Scale-Free-artig).\n"
        result += "   Künstliche Netze sind meist uniform.\n"
        result += "2. INHIBITION: Bio-Netze nutzen ~25% inhibitorische Neuronen.\n"
        result += "   Künstliche Netze haben keine echte Inhibition (nur negative Gewichte).\n"
        result += "3. RECIPROCITY: Bio-Netze haben partielle Rückkopplung.\n"
        result += "   Transformer: null. RNN: vollständig. Beides ist unbiologisch.\n"
        result += "4. SPARSITY: Bio-Netze sind extrem sparse mit lokaler Dichte.\n"
        result += "   Transformer-Attention ist dense. Das ist das Gegenteil.\n"
        result += "5. MODULARITÄT: Bio-Netze haben klare funktionale Module.\n"
        result += "   Transformer-Layer sind identisch und austauschbar.\n"
        
        return result
    
    # ========================================
    # TOOL 8: Pathway-Analyse
    # ========================================
    def find_pathways(self, source_region: str, target_region: str, n_paths: int = 5) -> str:
        """
        Findet die stärksten Signalwege zwischen zwei Gehirnregionen.
        """
        src_neurons = self.neurons[self.neurons["region"] == source_region]["neuron_id"].values
        tgt_neurons = set(self.neurons[self.neurons["region"] == target_region]["neuron_id"].values)
        
        if len(src_neurons) == 0 or len(tgt_neurons) == 0:
            return f"Region nicht gefunden! Verfügbar: {list(self.neurons['region'].unique())}"
        
        # Invertiere Gewichte für Dijkstra (stärkere Verbindung = kürzerer Pfad)
        G_inv = self.G.copy()
        for u, v, d in G_inv.edges(data=True):
            d["inv_weight"] = 1.0 / max(0.1, d.get("weight", 1))
        
        result = f"=== PATHWAYS: {source_region} → {target_region} ===\n\n"
        
        found_paths = []
        # Sample source neurons
        sample_src = np.random.choice(src_neurons, min(100, len(src_neurons)), replace=False)
        
        for src in sample_src:
            for tgt in list(tgt_neurons)[:50]:
                try:
                    path = nx.shortest_path(G_inv, src, tgt, weight="inv_weight")
                    # Pfadstärke = Minimum der Kantengewichte
                    path_strength = min(
                        self.G[path[i]][path[i+1]].get("weight", 1)
                        for i in range(len(path)-1)
                    )
                    found_paths.append((path, path_strength))
                except nx.NetworkXNoPath:
                    continue
            if len(found_paths) >= n_paths * 10:
                break
        
        # Top N stärkste Pfade
        found_paths.sort(key=lambda x: x[1], reverse=True)
        
        for i, (path, strength) in enumerate(found_paths[:n_paths]):
            result += f"Pfad {i+1} (Stärke={strength}, Länge={len(path)-1}):\n"
            for j, nid in enumerate(path):
                info = self.neurons[self.neurons["neuron_id"] == nid].iloc[0]
                arrow = "  → " if j > 0 else "    "
                result += f"{arrow}Neuron {nid} [{info['region']}/{info['cell_type']}/{info['neurotransmitter']}]\n"
            result += "\n"
        
        if not found_paths:
            result += "Keine direkten Pfade gefunden.\n"
        
        return result

    # ========================================
    # Tool-Registry für den Agenten
    # ========================================
    def get_tool_descriptions(self) -> str:
        """Gibt Beschreibungen aller Tools für das LLM zurück."""
        return """Available Analysis Tools:

1. find_hubs(top_n=20, metric="total_degree")
   Find the most important hub neurons. Metrics: total_degree, in_degree, out_degree, betweenness, pagerank

2. detect_communities(resolution=1.0)
   Find functional clusters using Louvain algorithm. Higher resolution = more smaller clusters.

3. trace_signal(start_neuron=ID, max_depth=5, min_weight=3)
   Trace signal propagation from a specific neuron through the network.

4. analyze_region(region="XX")
   Deep analysis of a brain region. Regions: MB, AL, OL, CX, LH, SEZ, VLNP, SLP, AVLP, GNG

5. find_motifs(sample_size=5000)
   Search for recurring network motifs (building blocks of neural circuits).

6. simulate_activation(start_neurons=[IDs], steps=10, threshold=0.3, decay=0.8)
   Simulate signal spreading through the network with excitation/inhibition.

7. compare_to_artificial()
   Compare biological network properties to Transformers, CNNs, and RNNs.

8. find_pathways(source_region="XX", target_region="YY", n_paths=5)
   Find strongest signal pathways between two brain regions.
"""

print("✅ BrainAnalyzer definiert (8 Tools)")

In [ ]:
# Analyzer initialisieren
analyzer = BrainAnalyzer(G, loader.neurons, loader.synapses)

# Schnelltest
print(analyzer.find_hubs(top_n=5, metric="pagerank"))

---
## 4. Der KI-Agent (Forscher)

Das Herzstück: Ein LLM das autonom das Gehirn erforscht.

In [ ]:
class NeuroAgent:
    """
    Autonomer Forschungsagent der das Connectome analysiert,
    Hypothesen bildet, und Prinzipien extrahiert.
    """
    
    SYSTEM_PROMPT = """Du bist ein Computational Neuroscience Forschungsagent.

Dein Ziel: Analysiere das Drosophila (Fruchtfliege) Connectome, finde biologische
Prinzipien der Informationsverarbeitung, und leite daraus Verbesserungen für
künstliche neuronale Netzwerke ab.

Du hast Zugriff auf ein vollständiges Connectome mit ~13.500 Neuronen in 10 Regionen.

{connectome_summary}

{tool_descriptions}

ARBEITSWEISE:
1. Analysiere systematisch - beginne mit der Gesamtstruktur, dann zoome rein
2. Bilde Hypothesen basierend auf deinen Beobachtungen
3. Teste Hypothesen mit den verfügbaren Tools
4. Dokumentiere Prinzipien die sich bestätigen
5. Leite konkrete Architektur-Vorschläge für KI ab

Wenn du ein Tool aufrufen willst, antworte EXAKT in diesem Format:
TOOL_CALL: tool_name(param1=value1, param2=value2)

Du kannst pro Antwort EINEN Tool-Call machen. Nach dem Ergebnis analysierst du
und entscheidest den nächsten Schritt.

Wenn du genug Informationen hast, schreibe:
FINDING: [Dein Prinzip/Erkenntnis]
ARCHITECTURE_PROPOSAL: [Konkreter Vorschlag für KI-Architektur]

Antworte auf Deutsch."""

    def __init__(self, analyzer: BrainAnalyzer, connectome_summary: str):
        self.analyzer = analyzer
        self.client = OpenAI(base_url=LLM_API_BASE, api_key=LLM_API_KEY)
        self.model = LLM_MODEL
        
        self.system_prompt = self.SYSTEM_PROMPT.format(
            connectome_summary=connectome_summary,
            tool_descriptions=analyzer.get_tool_descriptions(),
        )
        
        self.conversation = []
        self.findings = []          # Bestätigte Erkenntnisse
        self.hypotheses = []        # Aktive Hypothesen
        self.arch_proposals = []    # Architektur-Vorschläge
        self.step_count = 0
    
    def _call_llm(self, messages: list) -> str:
        """LLM API Call."""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.7,
                max_tokens=2000,
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"LLM ERROR: {e}"
    
    def _split_params(self, params_str: str) -> list:
        """Splittet Parameter-String korrekt, auch bei Listen wie [1, 2, 3]."""
        parts = []
        current = []
        depth = 0  # Klammer-Tiefe
        for ch in params_str:
            if ch == '[': depth += 1
            elif ch == ']': depth -= 1
            if ch == ',' and depth == 0:
                parts.append(''.join(current))
                current = []
            else:
                current.append(ch)
        if current:
            parts.append(''.join(current))
        return parts
    
    def _parse_tool_call(self, text: str) -> tuple:
        """Parst einen Tool-Call aus der LLM-Antwort."""
        for line in text.split("\n"):
            if line.strip().startswith("TOOL_CALL:"):
                call = line.split("TOOL_CALL:")[1].strip()
                # Parse: tool_name(param1=value1, param2=value2)
                try:
                    func_name = call.split("(")[0].strip()
                    params_str = call.split("(", 1)[1].rstrip(")")
                    params = {}
                    if params_str.strip():
                        for param in self._split_params(params_str):
                            key, val = param.split("=", 1)
                            key = key.strip()
                            val = val.strip().strip('"').strip("'")
                            # Typ-Konvertierung
                            try:
                                if val.startswith("["):
                                    val = json.loads(val)
                                elif "." in val:
                                    val = float(val)
                                else:
                                    val = int(val)
                            except (ValueError, json.JSONDecodeError):
                                pass  # String bleibt String
                            params[key] = val
                    return func_name, params
                except Exception as e:
                    return None, f"Parse error: {e}"
        return None, None
    
    def _execute_tool(self, func_name: str, params: dict) -> str:
        """Führt ein Analyse-Tool aus."""
        tool_map = {
            "find_hubs": self.analyzer.find_hubs,
            "detect_communities": self.analyzer.detect_communities,
            "trace_signal": self.analyzer.trace_signal,
            "analyze_region": self.analyzer.analyze_region,
            "find_motifs": self.analyzer.find_motifs,
            "simulate_activation": self.analyzer.simulate_activation,
            "compare_to_artificial": self.analyzer.compare_to_artificial,
            "find_pathways": self.analyzer.find_pathways,
        }
        
        if func_name not in tool_map:
            return f"Unknown tool: {func_name}. Available: {list(tool_map.keys())}"
        
        try:
            return tool_map[func_name](**params)
        except Exception as e:
            return f"Tool execution error: {e}"
    
    def _extract_findings(self, text: str):
        """Extrahiert Findings und Architecture Proposals."""
        for line in text.split("\n"):
            if line.strip().startswith("FINDING:"):
                finding = line.split("FINDING:")[1].strip()
                self.findings.append({
                    "step": self.step_count,
                    "finding": finding,
                    "timestamp": datetime.now().isoformat(),
                })
            if line.strip().startswith("ARCHITECTURE_PROPOSAL:"):
                proposal = line.split("ARCHITECTURE_PROPOSAL:")[1].strip()
                self.arch_proposals.append({
                    "step": self.step_count,
                    "proposal": proposal,
                    "timestamp": datetime.now().isoformat(),
                })
    
    def research_step(self, user_input: str = None) -> str:
        """
        Führt einen Forschungsschritt aus.
        Kann mit user_input gelenkt werden, oder autonom arbeiten.
        """
        self.step_count += 1
        
        # Kontext aufbauen
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(self.conversation[-20:])  # Letzte 20 Nachrichten als Kontext
        
        if user_input:
            messages.append({"role": "user", "content": user_input})
            self.conversation.append({"role": "user", "content": user_input})
        else:
            # Autonomer Schritt
            prompt = "Führe den nächsten logischen Forschungsschritt durch. "
            if self.findings:
                prompt += f"\n\nBisherige Erkenntnisse:\n"
                for f in self.findings:
                    prompt += f"- {f['finding']}\n"
            prompt += "\nWas willst du als nächstes untersuchen?"
            messages.append({"role": "user", "content": prompt})
            self.conversation.append({"role": "user", "content": prompt})
        
        # LLM aufrufen
        print(f"\n{'='*60}")
        print(f"🧠 FORSCHUNGSSCHRITT {self.step_count}")
        print(f"{'='*60}")
        
        response = self._call_llm(messages)
        print(f"\n🤖 AGENT:\n{response}")
        
        self.conversation.append({"role": "assistant", "content": response})
        
        # Tool-Call prüfen
        func_name, params = self._parse_tool_call(response)
        
        if func_name:
            print(f"\n🔧 TOOL: {func_name}({params})")
            tool_result = self._execute_tool(func_name, params)
            print(f"\n📊 ERGEBNIS:\n{tool_result[:2000]}")
            
            # Tool-Ergebnis zurück an LLM
            tool_msg = f"Tool-Ergebnis von {func_name}:\n\n{tool_result}"
            self.conversation.append({"role": "user", "content": tool_msg})
            
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": tool_msg})
            
            # LLM analysiert das Ergebnis
            analysis = self._call_llm(messages)
            print(f"\n🔬 ANALYSE:\n{analysis}")
            
            self.conversation.append({"role": "assistant", "content": analysis})
            self._extract_findings(analysis)
            
            return analysis
        
        self._extract_findings(response)
        return response
    
    def run_autonomous(self, n_steps: int = 5) -> list:
        """
        Führt N autonome Forschungsschritte durch.
        Die KI entscheidet selbst, was sie als nächstes untersucht.
        """
        print("🚀 STARTE AUTONOME FORSCHUNG")
        print(f"   Geplant: {n_steps} Schritte\n")
        
        results = []
        
        # Erster Schritt: Überblick
        result = self.research_step(
            "Beginne deine Forschung. Verschaffe dir zuerst einen Überblick über "
            "die Gesamtstruktur des Connectomes. Welches Tool willst du zuerst nutzen?"
        )
        results.append(result)
        
        # Folgeschritte: autonom
        for i in range(n_steps - 1):
            result = self.research_step()  # Kein Input = autonom
            results.append(result)
            time.sleep(1)  # Rate limiting
        
        return results
    
    def get_research_report(self) -> str:
        """Gibt einen Forschungsbericht zurück."""
        report = "\n" + "="*60 + "\n"
        report += "📋 FORSCHUNGSBERICHT\n"
        report += "="*60 + "\n\n"
        report += f"Schritte durchgeführt: {self.step_count}\n\n"
        
        report += "--- ERKENNTNISSE ---\n"
        for i, f in enumerate(self.findings, 1):
            report += f"  {i}. [Step {f['step']}] {f['finding']}\n"
        
        report += "\n--- ARCHITEKTUR-VORSCHLÄGE ---\n"
        for i, p in enumerate(self.arch_proposals, 1):
            report += f"  {i}. [Step {p['step']}] {p['proposal']}\n"
        
        if not self.findings:
            report += "  (Noch keine Erkenntnisse)\n"
        if not self.arch_proposals:
            report += "  (Noch keine Vorschläge)\n"
        
        return report

print("✅ NeuroAgent definiert")

---
## 5. Forschung starten

### Option A: Autonome Forschung (KI entscheidet selbst)

In [ ]:
# Agent erstellen
agent = NeuroAgent(analyzer, loader.summary())

# 5 autonome Forschungsschritte
results = agent.run_autonomous(n_steps=5)

In [ ]:
# Forschungsbericht
print(agent.get_research_report())

### Option B: Gelenkte Forschung (du gibst Richtung vor)

In [ ]:
# Beispiel: Gezielte Frage stellen
agent.research_step(
    "Untersuche die Verbindung zwischen der Mushroom Body (MB) Region und dem "
    "Central Complex (CX). Wie fließen Signale vom Gedächtnis zur Navigation?"
)

In [ ]:
# Weitere gelenkte Schritte
agent.research_step(
    "Vergleiche jetzt die biologischen Netzwerkeigenschaften mit Transformer-Architekturen. "
    "Was macht das biologische Netz anders, und was könnten wir übernehmen?"
)

### Option C: Interaktiver Modus

In [ ]:
# Interaktiver Forschungs-Loop
# Führe diese Zelle aus und gib Anweisungen ein (oder 'auto' für autonomen Schritt)

print("🧠 INTERAKTIVER FORSCHUNGSMODUS")
print("Befehle: Freitext = Anweisung, 'auto' = autonomer Schritt, 'report' = Bericht, 'quit' = Ende\n")

while True:
    user_input = input("\n>>> Deine Anweisung: ").strip()
    
    if user_input.lower() == "quit":
        print("\n👋 Forschung beendet.")
        break
    elif user_input.lower() == "report":
        print(agent.get_research_report())
    elif user_input.lower() == "auto":
        agent.research_step()
    else:
        agent.research_step(user_input)

---
## 6. Der Architekt — Von Prinzipien zu PyTorch-Code

Dieses Modul nimmt die Erkenntnisse des Forschers und generiert daraus
konkrete PyTorch-Module.

In [ ]:
class ArchitectGenerator:
    """
    Nimmt biologische Prinzipien und generiert daraus PyTorch-Module.
    Das ist der 'auf sich selbst anwenden'-Teil.
    """
    
    SYSTEM_PROMPT = """Du bist ein Neural Architecture Engineer.

Du bekommst biologische Prinzipien aus der Analyse eines echten Gehirns
und deine Aufgabe ist es, daraus KONKRETE PyTorch-Module zu generieren.

Regeln:
1. Jedes Modul muss valides PyTorch sein (import torch, torch.nn, etc.)
2. Jedes Modul muss eine forward()-Methode haben
3. Erkläre in Kommentaren, welches biologische Prinzip implementiert wird
4. Gib Empfehlungen, wo im Transformer dieses Modul eingefügt werden könnte
5. Gib den Code als vollständige Python-Datei aus

Antworte NUR mit dem Python-Code. Keine Erklärung außerhalb des Codes.
Starte mit ```python und ende mit ```."""

    def __init__(self):
        self.client = OpenAI(base_url=LLM_API_BASE, api_key=LLM_API_KEY)
        self.model = LLM_MODEL
        self.generated_modules = []
    
    def generate_module(self, findings: list, proposals: list) -> str:
        """
        Generiert ein PyTorch-Modul basierend auf den Forschungsergebnissen.
        """
        findings_text = "\n".join(f"- {f['finding']}" for f in findings)
        proposals_text = "\n".join(f"- {p['proposal']}" for p in proposals)
        
        prompt = f"""Basierend auf diesen biologischen Erkenntnissen aus dem Drosophila-Connectome:

ERKENNTNISSE:
{findings_text}

ARCHITEKTUR-VORSCHLÄGE:
{proposals_text}

Generiere ein PyTorch-Modul das die wichtigsten biologischen Prinzipien implementiert.
Das Modul soll als Drop-in Replacement oder Ergänzung für Standard-Transformer-Blöcke funktionieren."""

        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.3,
            max_tokens=4000,
        )
        
        code = response.choices[0].message.content
        
        # Code extrahieren
        if "```python" in code:
            code = code.split("```python")[1].split("```")[0]
        elif "```" in code:
            code = code.split("```")[1].split("```")[0]
        
        self.generated_modules.append({
            "code": code,
            "findings": findings_text,
            "timestamp": datetime.now().isoformat(),
        })
        
        return code
    
    def generate_default_module(self) -> str:
        """
        Generiert ein Standard-Modul basierend auf bekannten biologischen Prinzipien,
        auch wenn noch keine Forschung gelaufen ist.
        """
        # Hardcoded biologische Prinzipien als Fallback
        code = '''\
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class BioInspiredAttention(nn.Module):
    """
    Biologisch inspirierter Attention-Mechanismus.
    
    Implementiert drei Schlüsselprinzipien aus dem Drosophila-Connectome:
    
    1. SPARSE CONNECTIVITY: Statt Dense Attention werden nur die Top-K
       stärksten Verbindungen genutzt (wie im biologischen Netz,
       wo Konnektivitätsdichte < 0.01 ist).
    
    2. LATERAL INHIBITION: Aktive Neuronen hemmen ihre Nachbarn.
       Dies verbessert Signaltrennung und Kontrast.
       Im Bio-Netz: ~25% inhibitorische Neuronen.
    
    3. HETEROGENEOUS HUBS: Einige "Hub"-Neuronen haben überproportional
       viele Verbindungen (Scale-Free-Eigenschaft). Wir simulieren das
       durch lernbare Hub-Gewichte.
    """
    
    def __init__(self, d_model: int, n_heads: int = 8, top_k: int = 64,
                 inhibition_strength: float = 0.1, n_hubs: int = 16):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.top_k = top_k
        self.inhibition_strength = inhibition_strength
        
        # Standard Q, K, V Projektionen
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        # BIO-PRINZIP 2: Laterale Inhibition
        # Lernbare Inhibitions-Kernel (lokal)
        self.inhibition_kernel = nn.Parameter(
            torch.randn(n_heads, 1, 7) * 0.01  # 7er Fenster
        )
        
        # BIO-PRINZIP 3: Hub-Neuronen
        # Lernbare Hub-Positionen mit stärkeren Verbindungen
        self.hub_bias = nn.Parameter(torch.zeros(n_heads, n_hubs))
        self.hub_proj = nn.Linear(d_model, n_hubs)
    
    def forward(self, x, mask=None):
        B, T, D = x.shape
        
        # Q, K, V berechnen
        q = self.W_q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        # Attention Scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        # BIO-PRINZIP 1: Sparse Connectivity (Top-K)
        # Nur die stärksten K Verbindungen behalten
        if self.top_k < T:
            top_k_vals, top_k_idx = scores.topk(self.top_k, dim=-1)
            sparse_scores = torch.full_like(scores, float("-inf"))
            sparse_scores.scatter_(-1, top_k_idx, top_k_vals)
            scores = sparse_scores
        
        # BIO-PRINZIP 2: Laterale Inhibition
        # Benachbarte Attention-Scores hemmen sich gegenseitig
        if T > 7:
            # Jede Attention-Zeile als separates 1D-Signal behandeln
            # scores: [B, n_heads, T, T] -> [B*n_heads*T, 1, T]
            scores_flat = scores.reshape(B * self.n_heads * T, 1, T)
            # Kernel: [n_heads, 1, 7] -> für jeden Head einen Kernel
            # Expand: [1, 1, 7] broadcast über alle Batch*T Zeilen
            kernel = self.inhibition_kernel.mean(dim=0, keepdim=True)  # [1, 1, 7]
            inhibition = F.conv1d(scores_flat, kernel, padding=3)
            inhibition = inhibition.view(B, self.n_heads, T, T)
            scores = scores - self.inhibition_strength * inhibition
        
        # BIO-PRINZIP 3: Hub Bias
        # Hub-Neuronen bekommen stärkere Verbindungen
        hub_scores = self.hub_proj(x)  # [B, T, n_hubs]
        hub_attn = torch.einsum("bth,nh->bnt", hub_scores, self.hub_bias)
        # Hub-Attention auf alle Positionen verteilen
        hub_broadcast = hub_attn.unsqueeze(-1).expand(-1, -1, -1, T)
        scores = scores + 0.1 * hub_broadcast[:, :, :T, :]
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        
        return self.W_o(out)


class BioTransformerBlock(nn.Module):
    """
    Vollständiger Transformer-Block mit biologisch inspirierten Modifikationen.
    
    Zusätzliches Prinzip:
    4. MODULARE HETEROGENITÄT: Im Gegensatz zu Standard-Transformern,
       wo alle Layer identisch sind, hat das biologische Gehirn
       spezialisierte Regionen. Wir implementieren das durch
       lernbare Layer-spezifische Skalierungsfaktoren.
    """
    
    def __init__(self, d_model: int, n_heads: int = 8, d_ff: int = None,
                 top_k: int = 64, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or d_model * 4
        
        self.attention = BioInspiredAttention(d_model, n_heads, top_k)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feed-Forward mit E/I-Balance
        # 75% excitatory, 25% inhibitory (wie im Bio-Netz)
        exc_dim = int(d_ff * 0.75)
        inh_dim = d_ff - exc_dim
        
        self.ff_excitatory = nn.Sequential(
            nn.Linear(d_model, exc_dim),
            nn.GELU(),
        )
        self.ff_inhibitory = nn.Sequential(
            nn.Linear(d_model, inh_dim),
            nn.GELU(),
        )
        self.ff_out = nn.Linear(d_ff, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        # BIO-PRINZIP 4: Layer-spezifische Spezialisierung
        self.layer_scale = nn.Parameter(torch.ones(2))  # [attn_scale, ff_scale]
    
    def forward(self, x, mask=None):
        # Attention mit Residual
        attn_out = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out) * self.layer_scale[0]
        
        # Feed-Forward mit E/I-Balance
        normed = self.norm2(x)
        exc = self.ff_excitatory(normed)  # Erregendes Signal
        inh = self.ff_inhibitory(normed)  # Hemmendes Signal
        
        # Kombination: Erregung - Hemmung
        combined = torch.cat([exc, -inh], dim=-1)  # Inhibition ist negativ!
        ff_out = self.ff_out(combined)
        x = x + self.dropout(ff_out) * self.layer_scale[1]
        
        return x


# ============================================================
# QUICK TEST
# ============================================================
if __name__ == "__main__":
    model = BioTransformerBlock(d_model=512, n_heads=8, top_k=32)
    x = torch.randn(2, 128, 512)  # Batch=2, Seq=128, Dim=512
    out = model(x)
    print(f"Input:  {x.shape}")
    print(f"Output: {out.shape}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("\n✅ BioTransformerBlock funktioniert!")
'''
        return code

print("✅ ArchitectGenerator definiert")

### Architektur generieren

In [ ]:
architect = ArchitectGenerator()

# Option 1: Aus Forschungsergebnissen generieren (wenn der Agent schon geforscht hat)
if agent.findings:
    code = architect.generate_module(agent.findings, agent.arch_proposals)
    print("=== GENERIERTES MODUL (aus Forschung) ===")
    print(code)
else:
    # Option 2: Default Bio-Modul (basierend auf bekannten Prinzipien)
    code = architect.generate_default_module()
    print("=== BIO-INSPIRED TRANSFORMER MODULE ===")
    print(code)

In [ ]:
# Modul testen (braucht PyTorch)
try:
    exec(code)
except ImportError:
    print("⚠️ PyTorch nicht installiert. Installiere mit:")
    print("   pip install torch")
    print("\nDer Code ist aber syntaktisch korrekt und ready für deine GPU-Maschine.")

---
## 7. Visualisierung

In [ ]:
def plot_region_connectivity(neurons_df, synapses_df):
    """Visualisiert die Verbindungen zwischen Gehirnregionen."""
    # Region-zu-Region Konnektivität
    region_map = dict(zip(neurons_df["neuron_id"], neurons_df["region"]))
    
    conn = defaultdict(int)
    for _, row in synapses_df.iterrows():
        src_reg = region_map.get(row["pre_neuron"], "?")
        tgt_reg = region_map.get(row["post_neuron"], "?")
        if src_reg != "?" and tgt_reg != "?":
            conn[(src_reg, tgt_reg)] += row["n_synapses"]
    
    regions = sorted(neurons_df["region"].unique())
    matrix = np.zeros((len(regions), len(regions)))
    
    for (src, tgt), count in conn.items():
        i = regions.index(src)
        j = regions.index(tgt)
        matrix[i][j] = count
    
    # Normalisieren (log-scale für bessere Sichtbarkeit)
    matrix_log = np.log1p(matrix)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Heatmap
    im = axes[0].imshow(matrix_log, cmap="YlOrRd", aspect="auto")
    axes[0].set_xticks(range(len(regions)))
    axes[0].set_yticks(range(len(regions)))
    axes[0].set_xticklabels(regions, rotation=45, ha="right")
    axes[0].set_yticklabels(regions)
    axes[0].set_xlabel("Target Region")
    axes[0].set_ylabel("Source Region")
    axes[0].set_title("Region-zu-Region Konnektivität (log)")
    plt.colorbar(im, ax=axes[0])
    
    # Degree-Verteilung
    degrees = [d for _, d in G.degree()]
    axes[1].hist(degrees, bins=100, color="#2196F3", alpha=0.7, edgecolor="black", linewidth=0.3)
    axes[1].set_xlabel("Degree")
    axes[1].set_ylabel("Häufigkeit")
    axes[1].set_title("Degree-Verteilung (Scale-Free?)")
    axes[1].set_yscale("log")
    
    plt.tight_layout()
    plt.savefig(f"{DATA_DIR}/connectome_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 Gespeichert: {DATA_DIR}/connectome_analysis.png")

plot_region_connectivity(loader.neurons, loader.synapses)

In [ ]:
def plot_signal_simulation(analyzer, start_neurons, steps=15):
    """Visualisiert die Signalausbreitung über Zeit."""
    activation = defaultdict(float)
    for n in start_neurons:
        activation[n] = 1.0
    
    region_history = defaultdict(list)
    
    for step in range(steps):
        new_activation = defaultdict(float)
        for neuron, act in activation.items():
            if act < 0.3:
                continue
            for _, target, data in analyzer.G.out_edges(neuron, data=True):
                weight = data.get("weight", 1) / 20.0
                nt = data.get("type", "excitatory")
                if nt == "inhibitory":
                    new_activation[target] -= act * weight * 0.5
                else:
                    new_activation[target] += act * weight
            new_activation[neuron] += act * 0.8
        
        activation = defaultdict(float)
        for n, a in new_activation.items():
            activation[n] = max(0.0, min(1.0, a))
        
        # Pro Region zählen
        region_counts = defaultdict(int)
        for n, a in activation.items():
            if a > 0.3:
                row = analyzer.neurons[analyzer.neurons["neuron_id"] == n]
                if len(row) > 0:
                    region_counts[row.iloc[0]["region"]] += 1
        
        for reg in analyzer.neurons["region"].unique():
            region_history[reg].append(region_counts.get(reg, 0))
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    for reg, counts in sorted(region_history.items()):
        if max(counts) > 0:
            ax.plot(range(steps), counts, label=reg, linewidth=2)
    
    ax.set_xlabel("Zeitschritt")
    ax.set_ylabel("Aktive Neuronen")
    ax.set_title(f"Signalausbreitung von Neuronen {start_neurons}")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(f"{DATA_DIR}/signal_propagation.png", dpi=150, bbox_inches="tight")
    plt.show()

# Signal von der Mushroom Body starten
mb_neurons = loader.neurons[loader.neurons["region"] == "MB"]["neuron_id"].values[:5].tolist()
plot_signal_simulation(analyzer, mb_neurons)

---
## 8. Alles zusammen: Der volle Loop

```
Connectome laden → KI analysiert → Prinzipien finden → Architektur generieren → (Training)
```

In [ ]:
def full_research_pipeline(n_research_steps: int = 5):
    """
    Führt die komplette Pipeline durch:
    1. Connectome laden
    2. Graph bauen
    3. KI-Agent forscht autonom
    4. Architektur wird generiert
    5. Ergebnisse werden dokumentiert
    """
    print("🚀 NEUROAGENT — VOLLE PIPELINE")
    print("=" * 60)
    
    # 1. Daten
    print("\n📦 Phase 1: Daten laden...")
    loader = ConnectomeLoader(DATA_DIR)
    loader.download_flywire_sample()
    G = loader.build_graph()
    
    # 2. Analyzer
    print("\n🔬 Phase 2: Analyse-Tools bereit...")
    analyzer = BrainAnalyzer(G, loader.neurons, loader.synapses)
    
    # 3. Forschung
    print(f"\n🧠 Phase 3: Autonome Forschung ({n_research_steps} Schritte)...")
    agent = NeuroAgent(analyzer, loader.summary())
    agent.run_autonomous(n_steps=n_research_steps)
    
    # 4. Architektur
    print("\n🏗️ Phase 4: Architektur generieren...")
    architect = ArchitectGenerator()
    if agent.findings:
        code = architect.generate_module(agent.findings, agent.arch_proposals)
    else:
        code = architect.generate_default_module()
    
    # 5. Speichern
    output_path = f"{DATA_DIR}/bio_transformer.py"
    with open(output_path, "w") as f:
        f.write(code)
    print(f"\n💾 Architektur gespeichert: {output_path}")
    
    # Report
    print(agent.get_research_report())
    print("\n=== GENERIERTE ARCHITEKTUR ===")
    print(code[:1000] + "..." if len(code) > 1000 else code)
    
    return agent, architect, code

# Pipeline starten (auf 3 Schritte reduziert für Schnelltest)
# Für echte Forschung: n_research_steps=10 oder mehr
# agent, architect, code = full_research_pipeline(n_research_steps=3)

---
## 📋 Nächste Schritte

1. **LLM einrichten**: Starte Ollama/vLLM mit einem 70B+ Modell auf deiner GPU-Maschine
2. **Konfiguration anpassen**: `LLM_API_BASE` und `LLM_MODEL` oben setzen
3. **Autonome Forschung starten**: `full_research_pipeline(n_research_steps=10)`
4. **Echte Daten laden**: FlyWire Connectome von codex.flywire.ai herunterladen
5. **Training**: Generierte Module mit PyTorch auf Benchmark-Datensätzen testen

### FlyWire-Daten herunterladen (echtes Connectome)
```bash
# Voller Datensatz:
pip install caveclient
# Dann in Python:
from caveclient import CAVEclient
client = CAVEclient('flywire_fafb_production')
synapses = client.materialize.query_table('synapses_nt_v1')
```

### GPU-Training starten
```bash
# Auf deiner 8x RTX 6000 Maschine:
torchrun --nproc_per_node=8 train.py \
    --model bio_transformer \
    --data openwebtext \
    --batch_size 32
```